# Нейронные сети и обработка естественного языка - NLP

# Модуль 6. Использование API больших языковых моделей.

## 1. Ключи для доступа к API

**ВНИМАНИЕ**: для выполнения этого модуля вам понадобятся ключи для доступа к API. Если у вас их нет, пожалуйста, получите их заранее, следуя инструкциям ниже.


### 1.1 OpenAI API key

Мы начнем с использования OpenAI API, как с создателем стандарта API для доступа к большим языковым моделям. 

Получить ключ для OpenAI API можно по ссылке: https://platform.openai.com/settings/organization/api-keys

Если у вас нет возможности получить ключ для OpenAI API - посмотрите эту часть курса как лекцию, но примите участие в практических заданиях позже, с использованием других доступных API.


### 1.2 Sber GigaChat API key

Получить ключ для Sber GigaChat API можно по ссылке: https://developer.sber.ru/keys

### 1.3 Yandex AI Studio API key

Получить ключ для Yandex AI Studio API можно по ссылке: https://cloud.yandex.ru/ai-studio

### 1.4 OpenRouter API key

Получить ключ для OpenRouter API можно по ссылке: https://openrouter.ai/dashboard


### 1.6 Как хранить ключи для доступа к API

Можно использовать переменные окружения для хранения ключей доступа к API. Например, можно создать файл `.env` в корне проекта и добавить туда строки вида:

```
OPENAI_API_KEY=ваш_ключ_для_OpenAI
SBER_GIGACHAT_API_KEY=ваш_ключ_для_Sber_GigaChat
YANDEX_AI_STUDIO_API_KEY=ваш_ключ_для_Yandex_AI_Studio
OPENROUTER_API_KEY=ваш_ключ_для_OpenRouter
HF_TOKEN=ваш_HF_токен
MY_MODEL_API_KEY=ключ_для_какой-то_модели_в_локальной_сети
``` 

Этот метод может не работать в некоторых средах.

Более универсальное решение: создать файл `__config__.py` и хранить ключи там в виде переменных:

```python
OPENAI_API_KEY = "ваш_ключ_для_OpenAI"
SBER_GIGACHAT_API_KEY = "ваш_ключ_для_Sber_GigaChat"
YANDEX_AI_STUDIO_API_KEY = "ваш_ключ_для_Yandex_AI_Studio"
OPENROUTER_API_KEY = "ваш_ключ_для_OpenRouter"
HF_TOKEN = "ваш_HF_токен"
MY_MODEL_API_KEY = "ключ_для_какой-то_модели_в_локальной_сети"
```

И затем в любом месте проекта импортировать эти переменные:

```python
from __config__ import *
```

**ВНИМАНИЕ**: утечка (или компрометация) ключей доступа к API может привести к несанкционированному использованию вашего аккаунта, а значит к финансовым потерям или даже к полной блокировке аккаунта.

Файлы `.env` и `__config__.py` не должны попадать в систему контроля версий (например, Git). 

Для этого можно добавить их в файл `.gitignore`:
```
.env
__config__.py
```
Также НЕЛЬЗЯ хранить ключи доступа к API в виде открытого текста в коде, особенно если код будет публичным. 

Еще следует избегать попадания ключей в логи, отладочную информацию и даже в скринкасты.

## 2. Использование OpenAI API

### 2.1 Элементарное обращение к OpenAI API

In [ ]:
!pip install -q openai pandas tiktoken

In [ ]:
import os
from openai import OpenAI
from __config__ import OPENAI_API_KEY

from IPython.display import Markdown, display

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

client = OpenAI()
MODEL = "gpt-4.1-mini"

In [ ]:
# удобно использовать вот такую функцию
def ask(system_prompt, user_prompt, model=MODEL, temperature=0.3):
    response = client.chat.completions.create( # создаем чат-комплишн
        model=model, # указываем модель
        temperature=temperature, # температура
        top_p=0.9, # топ-p сэмплирование
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    
    print(response.choices[0].message.content)
    print("\nTOKENS:", response.usage)
    return response

system_prompt = "Ты опытный преподаватель Python. Отвечай кратко и понятно."

user_prompt = "Объясни, что такое API большой языковой модели."

response = ask(system_prompt, user_prompt)



`response` — это объект ответа API.
В нём лежит:

- сгенерированный текст;
- информация о модели;
- токены;
- причины остановки генерации;
- tool calls;
- logprobs;
- metadata.

In [ ]:
# ответ модели
response.choices[0].message.content

In [ ]:
# представить его в MD в jupyter notebook
display(Markdown(response.choices[0].message.content))

In [ ]:
# содержимое response в JSON:
print(response.model_dump_json(indent=2))

### 2.2. Системный промпт

В OpenAI API существуют три роли: `system`, `user` и `assistant`.

- `system` — это системный промпт, который задаёт общие правила для модели. Он может быть использован для указания роли модели, её целей, правил поведения и формата ответа.
- `user` — это пользовательский промпт, который содержит конкретный запрос или задачу для модели.
- `assistant` — это ответ модели на запрос пользователя.

Как может выглядеть системный промпт:

```python
system_prompt = """
Ты — [роль модели].

Цель:
- [что нужно сделать]

Правила:
- отвечай на русском языке;
- не выдумывай факты;
- если данных недостаточно, явно напиши об этом;
- соблюдай указанный формат ответа;
- не добавляй лишних комментариев.

Формат ответа:
[описание структуры ответа]

Ограничения:
- кратко;
- без markdown, если не попросят;
- только по переданным данным.
"""
```

In [ ]:
# для примера, снова былинное
system_prompt = """
Ты — сказитель древнерусских былин.

Правила стиля:
- отвечай в торжественном былинном стиле;
- используй архаизмы умеренно;
- сохраняй смысл технического объяснения;
- не превращай ответ в шутку;
- не используй слишком длинные предложения.
"""

user_prompt = "Объясни, что такое нейронная сеть."

response = ask(system_prompt, user_prompt, temperature=0.8)

#### 2.3.1. NER через OpenAI API

In [ ]:
document = """
Компания ООО Ромашка заключила договор с Иваном Петровым.
Поставка оборудования состоится 15 марта 2026 года в Санкт-Петербурге.
Сумма договора составляет 1 250 000 рублей.
Контактное лицо: Анна Смирнова, email: anna@example.com.
"""

system_prompt = """
Ты выполняешь NER — извлечение именованных сущностей из текста.

Нужно найти сущности типов:
- PERSON
- ORGANIZATION
- LOCATION
- DATE
- MONEY
- EMAIL

Правила:
- используй только данные из текста;
- не выдумывай сущности;
- верни только валидный JSON;
- без пояснений и markdown.

Формат:
{
  "entities": [
    {
      "text": "...",
      "type": "...",
      "normalized": "...",
      "confidence": 0.0
    }
  ]
}
"""

user_prompt = f"Извлеки сущности из документа:\n\n{document}"

response = ask(system_prompt, user_prompt)

#### 2.3.2. Машинный перевод через OpenAI API



In [ ]:
system_prompt = """
Ты — профессиональный переводчик.

Правила:
- переводи с русского на английский;
- сохраняй смысл, тон и структуру;
- технические термины переводи общепринято;
- не добавляй пояснений;
- если встречается неоднозначность, выбери наиболее вероятный вариант.
"""

user_prompt = """
Большие языковые модели можно использовать через API,
локально или как часть RAG-системы.
"""

response = ask(system_prompt, user_prompt)

#### 2.3.3. Суммаризация


In [ ]:
text = """
Transformer — архитектура нейронных сетей, основанная на механизме внимания.
Она стала основой современных больших языковых моделей. В отличие от RNN,
трансформеры лучше параллелятся и эффективнее работают с длинными зависимостями.
Также они используют позиционные кодировки для сохранения порядка слов в предложении.
Головы внимания позволяют модели фокусироваться на разных частях входного текста, что улучшает понимание контекста.
К тому же, трансформеры поддерживают масштабирование до миллиардов параметров, что значительно повышает их способность к генерации и пониманию сложного текста.
"""

system_prompt = """
Ты делаешь краткую суммаризацию текста.

Правила:
- не цитируй исходный текст;
- не добавляй новых фактов;
- сохрани главную мысль;
- ответь в 3 пунктах;
- язык ответа — русский.
"""

user_prompt = f"Суммаризируй текст:\n\n{text}"

response = ask(system_prompt, user_prompt)

#### 2.3.4. Анализ данных



In [ ]:
import pandas as pd

df = pd.DataFrame({
    "product": ["A", "B", "C", "D"],
    "revenue": [120000, 95000, 170000, 60000],
    "cost": [70000, 40000, 120000, 30000],
    "orders": [120, 80, 150, 45]
})

df

In [ ]:
table_text = df.to_csv(index=False)

system_prompt = """
Ты — аналитик данных.

Задача:
- проанализировать таблицу;
- найти лидеров и аутсайдеров;
- посчитать маржу, если возможно;
- сделать краткие бизнес-выводы.

Правила:
- используй только данные из таблицы;
- не выдумывай контекст;
- если расчёт невозможен, скажи почему;
- ответ структурируй: наблюдения, расчёты, вывод.
"""

user_prompt = f"Проанализируй CSV-таблицу:\n\n{table_text}"

response = ask(system_prompt, user_prompt)

### 2.2. Структурированный ответ

Можно сделать так, чтобы модель возвращала структурированный ответ в виде JSON, и более того, чтобы ответ укладывался в определенную схему.

In [ ]:
user_text = """
Здравствуйте! Меня зовут Иван Петров.
Хочу записаться на курс по NLP в июне.
Опыт программирования: Python, примерно 2 года.
Интересуют transformers, RAG и работа с API.
Телефон: +7 999 123-45-67.
"""

Мы хотим получить ответ в виде следующего JSON:
```json
{
  "person": {
    "full_name": "Иван Петров",
    "phone": "+7 999 123-45-67"
  },
  "course": {
    "topic": "NLP",
    "preferred_month": "июнь"
  },
  "background": {
    "programming_language": "Python",
    "experience_years": 2
  },
  "interests": [
    "transformers",
    "RAG",
    "работа с API"
  ],
}
```

In [ ]:
messages=[
        {
            "role": "system",
            "content": """
Ты извлекаешь структурированные данные из заявки на обучение.

Правила:
- используй только данные из текста;
- если поле неизвестно, верни пустую строку или 0;
- не добавляй поля вне схемы;
- не добавляй markdown;
- ответ должен строго соответствовать JSON Schema.
"""
        },
        {
            "role": "user",
            "content": user_text
        }
    ]

Задаем `json_schema`. 

Правила написания json_schema для OpenAI API:
1. `type` - тип данных (object, array, string, number, boolean)
2. `properties` - описание полей объекта (для type: object)
3. `items` - описание элементов массива (для type: array)
4. `required` - список обязательных полей (для type: object)
5. `description` - описание поля (необязательно, но полезно для понимания модели)
6. `enum` - список допустимых значений (для type: string или number)
7. `default` - значение по умолчанию (необязательно)

Подробнее - см. https://json-schema.org/

In [ ]:
json_schema = {
            "name": "course_request",
            "strict": True,
            "schema": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "person": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "full_name": {
                                "type": "string"
                            },
                            "phone": {
                                "type": "string"
                            }
                        },
                        "required": ["full_name", "phone"]
                    },
                    "course": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "topic": {
                                "type": "string"
                            },
                            "preferred_month": {
                                "type": "string"
                            }
                        },
                        "required": ["topic", "preferred_month"]
                    },
                    "background": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "programming_language": {
                                "type": "string"
                            },
                            "experience_years": {
                                "type": "number"
                            }
                        },
                        "required": [
                            "programming_language",
                            "experience_years"
                        ]
                    },
                    "interests": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },

                },
                "required": [
                    "person",
                    "course",
                    "background",
                    "interests",
                ]
            }
        }

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    response_format={
        "type": "json_schema",
        "json_schema": json_schema
    },
    messages=messages
)

raw_result = response.choices[0].message.content

print(raw_result)

In [ ]:
import json

json.loads(raw_result)

### 2.3. Потоковая генерация

Можно получать ответ от модели по частям, в виде потока данных. Это позволяет: 
- начать обработку ответа, не дожидаясь его полного формирования, 
- либо прекратить генерацию ответа досрочно (тем самым сократив расход токенов).



In [ ]:
stream = client.chat.completions.create(
    model="gpt-4.1-mini",
    stream=True,
    messages=[
        {
            "role": "user",
            "content": "Расскажи про transformers"
        }
    ]
)

for chunk in stream:
    delta = chunk.choices[0].delta.content

    if delta:
        print(delta, end="")

### 2.8 Тонкая настройка

In [ ]:
# temperature и top_p — это параметры, которые влияют на креативность и разнообразие ответов модели.
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    temperature=1.5,
    # top_p=0.5,
    messages=[
        {
            "role": "user",
            "content": "Придумай название космического стартапа"
        }
    ]
)

display(Markdown(response.choices[0].message.content))


In [ ]:
# max_tokens
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    max_tokens=30,
    messages=[
        {
            "role": "user",
            "content": "Подробно расскажи про attention"
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
# сидирование генерации
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    seed=42,
    temperature=0.8,
    messages=[
        {
            "role": "user",
            "content": "Придумай название нейросети"
        }
    ]
)
display(Markdown(response.choices[0].message.content))

### 2.9 Мультимодальность

Некоторые модели поддерживают работу не только с текстом, но и с изображениями, аудио и видео. Это позволяет создавать более сложные приложения, которые могут, например, описывать изображения, распознавать речь или анализировать видео.

In [ ]:
import base64

client = OpenAI()

with open("data/boats.jpg", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Что изображено?"
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{image_b64}"
                    }
                }
            ]
        }
    ]
)

display(Markdown(response.choices[0].message.content))

In [ ]:
display(response.usage.to_dict())

### 2.7. Расходы на API

Общую информацию о расходах на API можно найти в личном кабинете на сайте провайдера. 

Подсчитать количество токенов на запрос можно следующими способами:
1. Фактический расход токенов (запрос + ответ) можно получить из метаданных ответа API.
2. Для оценки количества токенов в запросе можно использовать специальные библиотеки, например `tiktoken` для OpenAI.

In [ ]:
usage = response.usage

print("Input tokens:", usage.prompt_tokens)
print("Output tokens:", usage.completion_tokens)
print("Total tokens:", usage.total_tokens)

Примерный подсчет токенов с помощью `tiktoken`:


In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4.1-mini")

text = """
Большие языковые модели принимают текст, разбивают его на токены
и генерируют ответ тоже токен за токеном.
"""

tokens = enc.encode(text)
print(tokens)

print("Примерный подсчет токенов с помощью `tiktoken`:", len(tokens))

**Мультимодальные данные** (например, изображения) могут потреблять токены по-разному в зависимости от модели и провайдера.

У OpenAI vision pricing обычно зависит от:

- размера изображения;
- количества тайлов;
- внутренних токенов изображения.

## 3. Sber GigaChat API

Для получения доступа к Sber GigaChat API необходимо:
- войти через Sber ID или Sber Business ID
- сгенерировать Authorization Key
- получить Acess Token. он действителен в течение 30 минут

<font size=+3>🤷‍♂️</font>


In [ ]:
import requests
import uuid

from __config__ import SBER_GIGACHAT_API_KEY

AUTH_KEY = SBER_GIGACHAT_API_KEY

import requests

url = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"

payload = 'scope=GIGACHAT_API_PERS'
headers = {
  'Content-Type': 'application/x-www-form-urlencoded',
  'Accept': 'application/json',
  'RqUID': str(uuid.uuid4()),
  'Authorization': f'Basic {AUTH_KEY}'
}

response = requests.request("POST", url, headers=headers, 
    data=payload, 
    verify=False    # важно! у них сертификат минцифры
    )

GIGACHAT_ACCESS_TOKEN = response.json()['access_token']

print(GIGACHAT_ACCESS_TOKEN)

In [ ]:
import httpx
from openai import OpenAI

http_client = httpx.Client(verify=False, timeout=60)  # важно! у них сертификат минцифры

client = OpenAI(api_key= GIGACHAT_ACCESS_TOKEN,
                base_url= "https://gigachat.devices.sberbank.ru/api/v1",
                http_client=http_client
                )


response = client.chat.completions.create(
    model="GigaChat",
    messages=[
        {
            "role": "user",
            "content": "Что такое transformers?"
        }
    ]
)

display(Markdown(response.choices[0].message.content))

## 4. Yandex AI Studio API

In [ ]:
from __config__ import YANDEX_AI_STUDIO_API_KEY

YANDEX_CLOUD_FOLDER = "b1g2d2fvo9gkfv62u8n6"
YANDEX_CLOUD_MODEL = "aliceai-llm/latest"

client = OpenAI(
    api_key=YANDEX_AI_STUDIO_API_KEY,
    base_url="https://ai.api.cloud.yandex.net/v1",
    project=YANDEX_CLOUD_FOLDER
)

response = client.chat.completions.create(
    model=f"gpt://{YANDEX_CLOUD_FOLDER}/{YANDEX_CLOUD_MODEL}",
    messages=[
        {"role": "user", "content": "Что такое transformers?"}
    ]
)


display(Markdown(response.choices[0].message.content))

In [ ]:
models = client.models.list()

for model in models.data:
    print(model.id)

## 5. OpenRouter API

OpenRouter API - это универсальный интерфейс для доступа к различным большим языковым моделям от разных провайдеров. Он позволяет использовать модели от OpenAI, Anthropic, Cohere, Hugging Face и других через единый API.

Для корректной работы с OpenRouter API необходимо зарегистрировать ваш API ключ к провайдеру, например, OpenAI.

In [ ]:
from __config__ import OPENROUTER_API_KEY

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

response = client.chat.completions.create(
    model="openai/gpt-4o",
    messages=[
        {
            "role": "user",
            "content": "Что такое transformers?"
        }
    ]
)

display(Markdown(response.choices[0].message.content))

**OpenRouter** также позволяет запускать бесплатные модели: https://openrouter.ai/openrouter/free

In [ ]:
# попробуем deepseek/deepseek-v4-flash
client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

response = client.chat.completions.create(
    model="deepseek/deepseek-v4-flash",
    messages=[
        {
            "role": "user",
            "content": "Что такое transformers?"
        }
    ]
)

display(Markdown(response.choices[0].message.content))


**ПРАКТИКА**: Для выбранного провайдера (OpenAI, Sber, Yandex, OpenRouter) попробуйте разные модели и сравните ответы на одни и те же вопросы. Обратите внимание на стиль ответа, полноту информации и соответствие вашим ожиданиям.

Выполните задачу NER для текста и сравните результаты.
```
Компания ООО «ТехИнновация» подписала контракт с ПАО «СеверЭнерго» 14 марта 2026 года. Документ был оформлен в Санкт-Петербурге менеджером Иваном Петровым. Общая сумма сделки составила 12 500 000 рублей. Для связи указаны email ivan.petrov@tech.ru и телефон +7 (921) 555-12-34.

В апреле 2026 года сотрудники Анна Смирнова и Алексей Волков отправились в Казань для участия в конференции AI Future 2026, организованной Университетом Иннополис. Во время мероприятия представители компании DeepVision Ltd. представили новую систему анализа документов на базе transformers и RAG.
```


In [ ]:
# ваш код здесь










## 6. API локальных моделей

На примере LLAMA.cpp:

```
./llama-server   \
-m ~/models/llm/unsloth/Qwen3.5-9B-GGUF/Qwen3.5-9B-UD-Q4_K_XL.gguf   \
--mmproj ~/models/llm/unsloth/Qwen3.5-9B-GGUF/mmproj-F16.gguf   \
-c 16384   \
--image-max-tokens 512   \
-ngl 999   \
-np 4   \
--host 0.0.0.0   \
--port 8080
```

In [ ]:
client = OpenAI(
    api_key="EMPTY",
    base_url="http://brainbox:8080/v1"
)

response = client.chat.completions.create(
    model="local-model",
    messages=[
        {
            "role": "user",
            "content": "Что такое transformers?"
        }
    ],
    temperature=0.3
)

Markdown(response.choices[0].message.content)

## 7. Роутинг запросов

Используя API, можно реализовать роутинг запросов к разным моделям в зависимости от их специализации. 

Например:
1. одна модель оценивает тип запроса
2. другие модели его выполняют




In [ ]:
user_text1 = """
Напиши краткое резюме этого текста:
Transformers replaced recurrent processing with attention mechanisms
and enabled efficient parallel training.
"""

user_text2 = """
Компания ООО Ромашка заключила договор с Иваном Петровым.
Поставка оборудования состоится 15 марта 2026 года в Санкт-Петербурге.
Сумма договора составляет 1 250 000 рублей.
Контактное лицо: Анна Смирнова, email: anna@example.com.
"""

user_text3 = """
Расскажи что такое трансформеры простыми словами, как для ребёнка 10 лет.
"""

In [ ]:
from openai import OpenAI
import os
import json

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

client = OpenAI()
MODEL = "gpt-4.1-mini"

# ============================================
# 1. ROUTER: определяем тип задачи
# ============================================

user_text = user_text1  # попробуйте менять на user_text1 и user_text3

router_system_prompt = """
Ты — router для пользовательских запросов.

Определи тип задачи.

Возможные task_type:
- translation
- summarization
- ner
- table_analysis
- general_question

Верни только JSON:
{
  "task_type": "...",
  "language": "ru/en/fr/de/etc",
  "confidence": 0.0
}

Правила:
- не выполняй задачу;
- только классифицируй;
- JSON должен быть валидным.
"""

router_response = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": router_system_prompt},
        {"role": "user", 
            "content": user_text,
            }
    ]
)

route = json.loads(router_response.choices[0].message.content)

print("ROUTER RESULT:")
print(json.dumps(route, ensure_ascii=False, indent=2))




In [ ]:
# ============================================
# 2. Выбираем новый system prompt
# ============================================

system_prompts = {
    "translation": """
Ты — профессиональный переводчик.

Если текст не на русском — переведи на русский.
Если текст уже на русском — верни без изменений.

Не добавляй пояснений.
""",

    "summarization": """
Ты — сервис суммаризации.

Сделай краткое резюме текста на русском языке.

Формат:
1. Краткое резюме
2. Главная мысль
3. Ключевые термины

Не добавляй фактов, которых нет в тексте.
""",

    "ner": """
Ты — сервис NER.

Извлеки сущности:
- PERSON
- ORGANIZATION
- LOCATION
- DATE
- MONEY
- EMAIL

Верни только JSON:
{
  "entities": [
    {
      "text": "...",
      "type": "...",
      "normalized": "..."
    }
  ]
}
""",

    "table_analysis": """
Ты — аналитик данных.

Проанализируй таблицу или CSV.

Ответ:
1. Что есть в данных
2. Главные наблюдения
3. Возможные аномалии
4. Вывод

Используй только переданные данные.
""",

    "general_question": """
Ты — преподаватель по NLP и LLM.

Отвечай кратко, понятно и на русском языке.
"""
}

task_type = route.get("task_type", "general_question")

if task_type not in system_prompts:
    task_type = "general_question"

selected_system_prompt = system_prompts[task_type]

print("\nSELECTED TASK TYPE:")
print(task_type)

In [ ]:

# ============================================
# 3. Второй API-вызов с новым system prompt
# ============================================

final_response = client.chat.completions.create(
    model=MODEL,
    temperature=0.3,
    messages=[
        {"role": "system", "content": selected_system_prompt},
        {"role": "user", "content": user_text}
    ]
)

answer = final_response.choices[0].message.content

print("\nANSWER:")
print(answer)